# Атака на подгруппы в протоколе Диффи-Хеллмана

## Короткое введение
Протокол Диффи-Хеллмана дает возможность двум сторонам обменяться ключевой информацией так, что любой неактивный (без возможности влиять на систему) злоумышленник не может получить эту информацию.

Высокоуровневое представление DH:

1. Алиса создает закрытый ключ $a$, открытый ключ $A$, отправляет $A$ Бобу

2. Боб создает закрытый ключ $b$, открытый $B$, отправляет $B$ Алисе

3. Алиса и Боб вырабатывают новый (одинаковый) ключ из $(a,B)$ и $(b,A)$. 


Любой неактивный злоумышленник получает открытый ключи $(A,B)$, но вычисление $(a,B)=(b,A)=>K$ на их основе - задача в NP.

## Пример
Один из способов генерации таких пар $(a,A)$, $(b,B)$ - использование мультипликативных групп по модулю простого числа (обычно обозначается $Z^{*}_{p}$).
$Z^{*}_{p}$ состоит из чисел от $1$ до $p-1$, т.е. общее число элементов или мощность группы $Z^{*}_{p}$ равна $p-1$.

Посмотрим на пример: $Z^{*}_{5}$

Таблица умножения:

|   | 1 |2  | 3 | 4 |
|---|---|---|---|---|
| **1**  | 1  | 2  | 3  | 4  |
| **2** | 2  | 4  | 1  | 3  |
| **3**  | 3  | 1  | 4  | 2  |
| **4**  | 4  | 3  | 2  | 1  |

Заметно, что:

+ 1 - нейтральный элемент (любой элемент, умноженный на 1 остается тем же элементом)

+ 2 - обратный элемент к  3, а 4 - сам к себе

Также можно заметить, что это циклическая группа (что нормально для $Z^{*}_{p}$), значит можно получить все элементы группы последовательным умножением одного элемента (генератора) на себя.  В данном случае один из возможных генераторов - элемент $2$:

+ $2=2$

+ $2*2=4$

+ $4*2=3$

+ $3*2=1$

Так каким образом работает DH в $Z^{*}_{p}$?
Об используемых простом числе $p$ и генераторе $g$ договариваются заранее (обычно они часть стандарта).

1. Алиса и Боб выбирают случайные $a,b \in {2,p-2}$ (Алиса выбирает $a$, Боб выбирает $b$)

2. Алиса вычисляет $A=g^{a}\  mod\  p$, Боб вычисляет $B=g^{b}\  mod \  p$

3. Они обмениваются $A$ и $B$

4. Алиса вычисляет $C=B^{a}\  mod\  p=g^{ab}\  mod \  p$

5. Боб вычисляет $C=A^{b}\  mod \  p=g^{ba}\  mod \  p$

## Проблемы
В общем случае злоумышленнику сложно получить такое $a=log(A)$, что $A=g^{a} mod\ p$, но есть исключения.
Мультипликативные группы по модулю простого числа содержат подгруппы (вообще группы содержат подгруппы). В лучшем случае кроме самой группы существует только три подгруппы: ${1}, {1,p-1}$  и ещё одна подгруппа мощностью $\frac{p-1}{2}$. Такое возможно, когда $p$ - безопасное простое число ($\frac{p-1}{2}$ тоже простое).

Для каждого делителя $d$ ($d|p-1$) порядка (мощности) группы существует подгруппа $H$ группы $Z^{*}_{p}$ ($H \lt Z^{*}_{p}$), такая что её порядок равен $d$ ($|H| = d$). Например, в случае $p=13$, $p-1=12$ делится на ${1,2,3,4,6,12}$
Поэтому есть подгруппы степеней $2, 3, 4, 6$
В данном случае $2$ - генератор.

Давайте посмотрим на генерируемую циклическую группу:

In [1]:
p=13
for i in range(p):
    print("2^{0}={1}".format(i,pow(2,i,p)))

2^0=1
2^1=2
2^2=4
2^3=8
2^4=3
2^5=6
2^6=12
2^7=11
2^8=9
2^9=5
2^10=10
2^11=7
2^12=1


Как же найти генератор для подгруппы заданного размера? Это очень легко.  Для подгруппы порядка $d$, генератором будет $g_{d}=g^{\frac{p-1}{d}} \  mod \  p$.
Логично, что при возведении $g_{d}$ в степень $d$ результат равен $g^{d}_{d}=g^{\frac{p-1}{d}*d} \  mod \space p=g^{p-1}\  mod \  p=1 \  mod \  p$

Мы можем выразить закрытый ключ $a$ как $a=k*d+r$, где $r<d$, $k \in N$. Если мы возведём открытый ключ $A=g^a \  mod \  p$ в степень $\frac{p-1}{d}$, результат будет равен $A^{\frac{p-1}{d}}\  mod \  p=g^{a*\frac{p-1}{d}}\  mod \  p=g^{(k*d+r)*\frac{p-1}{d}}\  mod \  p=g^{k*d*\frac{p-1}{d}+r*\frac{p-1}{d}}\  mod \  p=g^{k*(p-1)+r*\frac{p-1}{d}}\  mod \  p=g^{r*\frac{p-1}{d}}\  mod \  p$.

Сравнивая $A^{\frac{p-1}{d}}\  mod \  p$ со всеми возможными $g^{l}_{d}\  mod \  p$, $l \in \{0,...,d-1\}$, мы можем найти $r$.

Давайте попробуем:

In [2]:
a=7
g=2
A=pow(g,a,p)
d=4
g_d=pow(g,(p-1)//d,p)
Ap=pow(A,(p-1)//d,p)
print ('A^{0}={1}'.format((p-1)//d,Ap))
for i in range(d):
    cur_pow=pow(g_d,i,p)
    if Ap!=cur_pow:
        print('(g_{0})^{1}={2}!=A^((p-1)/d)'.format(d,i,pow(g_d,i,p)))
    else:
        print('(g_{0})^{1}={2}=A^((p-1)/d)'.format(d,i,pow(g_d,i,p)))
        print ('a={0} mod {1}'.format(i,d))

A^3=5
(g_4)^0=1!=A^((p-1)/d)
(g_4)^1=8!=A^((p-1)/d)
(g_4)^2=12!=A^((p-1)/d)
(g_4)^3=5=A^((p-1)/d)
a=3 mod 4


Таким образом достаточно просто восстановить пары $r_{i}\equiv a \  (mod\  d_{i})$ для $p=13$.
Каким образом получить $a$ из $r_{i}$?
Китайская Теорема об Остатках гласит, что это возможно, если все $d_{i}$ взаимно простые и их произведение превышает $a$ (т.е. нам надо использовать взаимно простые делители).

In [18]:
try:
    from gmpy2 import invert
except ImportError:
    try:
        from Crypto.Util.number import inverse as invert
    except ImportError:
        print ("You need to install either gmpy2:\nsudo apt install python-gmpy2\nor pycryptodome:\npython3 -m pip install pycryptodome")
        raise Exception
def crt(remainders,modules,M):
    result = 0
    for (a, b) in zip(remainders,modules):
        result = (result+a*((M)//b)*invert((M)//b, b)) % (p-1)
    return result
    

Давайте попробуем для $a=7$, $p=13$ <br>
Мы знаем, что:

+ $a\equiv 3\  mod\  4$

+ $a\equiv 1\  mod\  3$


In [4]:
remainders=(1,3)
modules=(3,4)
m=p-1
print ('a={0}'.format(crt(remainders,modules,m)))

a=7


В общем случае мы можем разложить $p-1=p^{k_1}_1*...*p^{k_n}_n$ 

Случай, когда все $p_i$ малы ,а $k_i=1$, самый простой. Но что если все $p_i$  малы, но есть несколько $k_i\neq1$, такие что прямой перебор подгруппы порядка $p^{k_i}_i$ занимает слишком много времени?

Сначала надо решить $a \equiv r\  mod \  p_i$.

Теперь нам известно, что $a=k*p_i+r$, $k \in \{0\}\cup N$, где $k$ - неизвестно. Нам надо найти $a=r_1\  mod \  p^2_i$

Можно переписать выражение как $a=k_1*p^2_i+k_2*p_i+r$, $k_2 \in \{0,...,p_i-1\}$, $k_1 \in \{0\}\cup N$ 

$r_1=k_2*p_i+r$

Давайте посмотрим, что происходит при возведении $A$ в $\frac{p-1}{p^2_i}$ 

$A^{\frac{p-1}{p^2_{i}}}\ mod\ p=g^{a*\frac{p-1}{p^2_{i}}}\ mod\ p=g^{(k_1*p^2_i+k_2*p_i+r)*\frac{p-1}{p^2_{i}}}\ mod\ p=g^{k_1*p^2_i*\frac{p-1}{p^2_{i}}+k_2*p_i*\frac{p-1}{p^2_{i}}+r*\frac{p-1}{p^2_{i}}}\ mod\ p=g^{k_1*(p-1)+k_2*\frac{p-1}{p_{i}}+r*\frac{p-1}{p^2_{i}}}\ mod\ p=g^{k_2*\frac{p-1}{p_{i}}+r*\frac{p-1}{p^2_{i}}}\ mod\ p$.

Значит $A^{\frac{p-1}{p^2_{i}}}\ mod\ p=g^{k_2*\frac{p-1}{p_{i}}+r*\frac{p-1}{p^2_{i}}}\ mod\ p=g^{k_2*\frac{p-1}{p_{i}}}*g^{r*\frac{p-1}{p^2_{i}}}\ mod\ p$.

Единственная неизвестная - это $k_2$, которая определена в небольшом интервале и её можно перебрать.

Рассмотрим пример для $p=19$: 

In [5]:
import random
p=19
a= random.randint(2,p-2)
g=2
A=pow(g,a,p)
print ("Initial a={0},A={1}".format(a,A))
d=3
A_d=pow(A,(p-1)//d,p)
g_d=pow(g,(p-1)//d,p)
print('g^{0}={1}'.format((p-1)//d,g_d))
x=1
r_d=-1
for i in range(d):
    
    if x==A_d:
        r_d=i
        break
    x=(x*g_d)%p
print ('a={0} mod {1}'.format(r_d,d))

d2=d*d
A_d2=pow(A,(p-1)//d2,p)
g_d2=pow(g,(p-1)//d2,p)
g_needed=pow(g_d2,d,p)

x=pow(g_d2,r_d,p)
for i in range(d):
    if x==A_d2:
        k_d2=i
        break
    x=(x*g_needed)%p
r_d2=k_d2*d+r_d
print ('a={0} mod {1}'.format(r_d2,d2))


Initial a=17,A=10
g^6=7
a=2 mod 3
a=8 mod 9


Теперь мы знаем такие $r$, что $a\equiv r\  mod\  9$<br>
Всё, что осталось сделать, это найти $r$ для $a\equiv r\  mod\  2$ <br>

In [6]:
remainders=[r_d2]
modules=[d2]
d=2
A_d=pow(A,(p-1)//d,p)
g_d=pow(A,(p-1)//d,p)
x=1
for i in range(d):
    if x==A_d:
        r=i
        break
    x=(x*g_d)%p
remainders.append(r)
modules.append(d)
print ('Computed a={0}'.format(crt(remainders,modules,p-1)))

Computed a=17


## Задание
Теперь вы готовы решить задание. Можете воспользоваться API для общения с сервером или подсоединиться к нему при помощи netcat.
Сервер выдаст вам модуль и открытый ключ $A$. Используемый генератор - $2$. Ваша цель - найти закрытый ключ $a$.
Сначала нужно разложить $p-1$ на множители (все простые множители меньше $65536$), потом отобразить $A$ в подгруппы, как мы это делали ранее. Удачи!

In [59]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1345))
       
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения от сервера, по умолчанию до приглашения"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу.')
            return (None,None)
        if show:
            print (data)
        p=int(re.search(r'(?<=p=)\d+',data).group(0))
        A=int(re.search(r'(?<=2\*\*k \(mod p\)=)\d+',data).group(0))
        return (p,A)
    
    def checkSolution(self,k, show=True):
        self.s.sendall((str(k)+'\n').encode())
        data=self.recv_until(b'\n')
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу.')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        else:
            data=self.recv_until(b'>')
            try:
                data=data.decode()
            except UnicodeDecodeError:
                print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу.')
                return None
            if show:
                print (data)
            return False
    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(p,A)=vs.getChallenge()
vs.checkSolution(1)

Welcome to Diffie-Hellman subgroup task
p=23042411772043773768618112743136333261599072876686202517030871688003626524431197273685219763723918876609362171357788597254378871758239412276137802314155558688147287197670513476016727011391763553572922858713749745174396353258406902710032635762059821503904299690651692459166551217193283027640399402676766057638250166648587900927964075801984880058193238368073604643230258871995663081999548276618403866989119801783060591214405743060136510435339254508081542825211998259313005971117585068963688554858827655326353286635405252953250047402595061477073552332199663933061840738316761738346815999888244679924481075488301376607667
2**k (mod p)=195431759382001319205096455288884080041465983430578725706900362875786419922905190001966499319597447701157855793823363734770229218015664036579854083556876333365166327646427738990015750086420496365732329449432910097212682493401934517844507187675701372995636484281738315933608184218994264629809697646849955492627063532493536980253

False

1. Получаем простые делители и и их степени

In [60]:
print(f"p.bit_length(): {p.bit_length()}")

p.bit_length(): 2048


In [63]:
def factorize(n, limit=65536):
    def generate_primes(limit):
        sieve = [True] * (limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(limit**0.5) + 1):
            if sieve[i]:
                for j in range(i * i, limit + 1, i):
                    sieve[j] = False
        return [i for i, is_prime in enumerate(sieve) if is_prime]

    primes = generate_primes(limit)
    factors = {}
    remaining = n

    for q in primes:
        if remaining % q == 0:
            e = 0
            while remaining % q == 0:
                remaining //= q
                e += 1
            factors[q] = e

    if remaining > 1:
        if remaining < limit**2:
            factors[remaining] = 1
        else:
            pass

    product = 1
    for q, e in factors.items():
        product *= q**e

    if product != n:
        print(f"  Error factorizing")
    else:
        print(f"  Fcatorized!")

    return factors

Итак, разложили число $p - 1$ на степени простых чисел. Это исходный порядок группы и его делители.

2. Реализация алгоритма Полига-Хеллмана для поиска $ a\mod q^{e}$

In [64]:
def pohlig_hellman_prime_power(g, A, p, q, e):
    order = p - 1
    q_e = q ** e
    m = order // q_e

    g0 = pow(g, m, p)
    A0 = pow(A, m, p)

    h = pow(g0, q_e // q, p)

    x = 0
    q_pow = 1

    for k in range(e):
        g0_inv_x = pow(g0, (q_e - x) % q_e, p)
        beta = (A0 * g0_inv_x) % p

        exp = q_e // (q_pow * q)
        beta_prime = pow(beta, exp, p)

        y_found = None
        cur = 1
        for y in range(q):
            if cur == beta_prime:
                y_found = y
                break
            cur = (cur * h) % p

        x = (x + y_found * q_pow) % q_e
        q_pow *= q

    print(f"    a mod {q}^{e} = {x}")
    return x

3. Далее, реализуем КТО

In [65]:
def crt(remainders, moduli, total_modulus=None):

    M = 1
    for m in moduli:
        M *= m

    if total_modulus and total_modulus != M:
        print(f"  Внимание: total_modulus={total_modulus} != M={M}")

    result = 0
    for r, m in zip(remainders, moduli):
        Mi = M // m
        inv = invert(Mi, m)
        result = (result + r * Mi * inv) % M

    if total_modulus:
        result %= total_modulus

    return result

4. Собираем всё вместе

In [66]:
def diffie_hellman_attack(p, A):

    n = p - 1
    factors = factorize(n)

    product = 1
    for q, e in factors.items():
        product *= q ** e

    remainders = []
    moduli = []

    for q, e in factors.items():
        r = pohlig_hellman_prime_power(2, A, p, q, e)

        remainders.append(r)
        moduli.append(q ** e)

    a = crt(remainders, moduli, total_modulus=n)

    if pow(2, a, p) == A:
        print(f"Check passed: 2^{a} mod p = {A}")

    return a

5. Запуск атаки

In [68]:
def run_attack():
    client = VulnServerClient()

    while True:

        p, A = client.getChallenge()
        if p is None or A is None:
            break

        k = diffie_hellman_attack(p, A)

        if client.checkSolution(k):
            print(f"\nDone!")
            break

run_attack()

Welcome to Diffie-Hellman subgroup task
p=23042411772043773768618112743136333261599072876686202517030871688003626524431197273685219763723918876609362171357788597254378871758239412276137802314155558688147287197670513476016727011391763553572922858713749745174396353258406902710032635762059821503904299690651692459166551217193283027640399402676766057638250166648587900927964075801984880058193238368073604643230258871995663081999548276618403866989119801783060591214405743060136510435339254508081542825211998259313005971117585068963688554858827655326353286635405252953250047402595061477073552332199663933061840738316761738346815999888244679924481075488301376607667
2**k (mod p)=177731939674591421996786655528332928971909952867395541611994916716967815853730662374248199143958366154095806580271877113780069418949937395385294525543082882892380351259885009530874258780148150217759971771109635449778103524437775358820899305609604027808189756060910552135840496253390167457984689298726083835195556557979226902915